In [ ]:
import math
import matplotlib.pyplot as plt
import numpy as np
from unit_conversions import Length, Weight, Speed, Force

In [ ]:
inch_to_m = 0.0254  # [=] m/inch
feet_to_m = 0.3048  # [=] m/ft
lb_to_kg = 0.453592  # [=] kg/lb
g = 9.81  # [=] m/s^2

bank_angles = np.linspace(1, 80, 80)
g_loadings = 1 / np.cos(np.deg2rad(bank_angles))

v = 100 * feet_to_m
Rs = v**2 / g / np.tan(np.deg2rad(bank_angles))
turning_time = 2 * np.pi * Rs / v

# Make two y axes
fig, g_load_axis = plt.subplots()
g_load_axis.plot(bank_angles, g_loadings, label="G loading", color="tab:orange")
g_load_axis.set_xlabel("Bank angle (°)")
g_load_axis.set_ylabel("G Loading")

turn_radius_axis = g_load_axis.twinx()
turn_radius_axis.plot(bank_angles, Rs, label="Turn Radius")
turn_radius_axis.set_ylabel("Turn radius")
turn_radius_axis.set_ylim([0, 500])

# plt.plot(bank_angles, g_loadings, label= "G Loading")
# plt.plot(bank_angles, Rs)
# plt.plot(bank_angles, Rs)

# plt.xlabel('Bank angle [deg]')
# plt.ylabel('g loading')
fig.legend()
plt.show()

# Mission 2
See [the initial analysis](initial_analysis.ipynb) for scoring rules

In [ ]:
# --- Constants based on the DBF Rules---

# Passenger (Duck) Properties
DUCK_WEIGHT_OZ = 0.7  # Using max weight for a conservative estimate (0.6-0.7 oz)
DUCK_WIDTH_IN = 2.3
DUCK_LENGTH_IN = 2.5
DUCK_HEIGHT_IN = 2.5
DUCK_VOLUME_IN3 = DUCK_WIDTH_IN * DUCK_LENGTH_IN * DUCK_HEIGHT_IN

# Cargo (Hockey Puck) Properties
PUCK_WEIGHT_OZ = 6.0
PUCK_DIAMETER_IN = 3.0
PUCK_THICKNESS_IN = 1.0
PUCK_VOLUME_IN3 = math.pi * ((PUCK_DIAMETER_IN / 2) ** 2) * PUCK_THICKNESS_IN

# Mission 2 Scoring Parameters
INCOME_FIXED_PER_PASSENGER = 6
INCOME_PER_PASSENGER_PER_LAP = 2
INCOME_FIXED_PER_CARGO = 10
INCOME_PER_CARGO_PER_LAP = 8

COST_BASE_PER_LAP = 10
COST_PER_PASSENGER_PER_LAP = 0.5
COST_PER_CARGO_PER_LAP = 2

In [ ]:
def math_max(x, y):
    # Literally just a "max" function but to avoid errors with arrays
    return(
        np.heaviside(x - y, 0) * (x-y) +
        y
    )

In [ ]:
def payload_properties(num_passengers, num_cargo=None):
    """
    Calculates the total weight and estimated volume for passengers and associated cargo.

    Args:
        num_passengers (int): The number of passengers (ducks).

    Returns:
        dict: A dictionary containing the number of cargo pieces, total weight in ounces,
              and estimated total volume in cubic inches. Returns None if num_passengers < 3.
    """
    # Although the plane cargo capacity must be set to the following
    # constraints, Mission 2 loading does not have any constraints
    
    # We are not checking for any rule violations. Good luck!

    # if num_passengers.any() < 3:
    #     # Although the plane must be *capable* of carrying 3, a flight may have 0.
    #     '''
    #     raise Warning(
    #         "DBF RULE VIOLATION: MINIMUM PASSENGER COUNT\nThere must be a minimum of 3 passengers. {num_passengers} passengers requested"
    #     )
    #     '''
    
    # if num_cargo is None:
    #     # If not cargo is specified assume max possible cargo loading
    #     num_cargo = np.floor(num_passengers / 3)
    # elif num_cargo * 3 > num_passengers:
    #     '''
    #     raise Warning(
    #         "DBF RULE VIOLATION: CARGO-PASSENGER RATIO\nThere must be a minimum of {} passengers for {} cargo. {} cargo and {} passengers requested".format(
    #             num_cargo * 3,
    #             num_cargo,
    #             num_cargo,
    #             num_passengers,
    #         )
    #     )
    #     '''

    # Minimum design constraints by volume
    min_passenger_volume = math_max(
        3 * DUCK_VOLUME_IN3, # Minimum 3 passengers
        num_cargo * 3 * DUCK_VOLUME_IN3, # Minimum 3 passengers per cargo
    )

    total_weight_oz = (
        (num_passengers * DUCK_WEIGHT_OZ) + (num_cargo * PUCK_WEIGHT_OZ)
    )

    total_volume_in3 = (
        math_max((num_passengers * DUCK_VOLUME_IN3), min_passenger_volume) +
        (num_cargo * PUCK_VOLUME_IN3)
    )

    return {
        "num_pax": num_passengers,
        "num_cargo": num_cargo,
        "weight_oz": total_weight_oz,
        "estimated_volume_in3": total_volume_in3,
    }

In [ ]:
def net_income(
    num_passengers,
    num_cargo,
    num_laps,
    battery_capacity_wh=100.0,
):
    """
    Calculates the Net Income for Mission 2 based on the rules.

    Args:
        num_passengers (int): The number of passengers (ducks) flown.
        num_laps (int): The number of laps completed.
        battery_capacity_wh (float, optional): The total propulsion battery capacity in Watt-hours. Defaults to 100.0.

    Returns:
        float: The calculated Net Income for the mission.
    """
    # The minimum capacity of the airplane is 3 passengers and 1 cargo.
    # However, a team can choose to fly with fewer or no payload.
    payload_props = payload_properties(num_passengers, num_cargo)
    num_cargo = payload_props["num_cargo"] * np.heaviside(num_passengers, 0)

    # Calculate total income
    income = (num_passengers * (INCOME_FIXED_PER_PASSENGER + (INCOME_PER_PASSENGER_PER_LAP * num_laps))) + (num_cargo * (INCOME_FIXED_PER_CARGO + (INCOME_PER_CARGO_PER_LAP * num_laps)))

    # Calculate total cost
    efficiency_factor = battery_capacity_wh / 100.0
    cost_per_lap = COST_BASE_PER_LAP + (num_passengers * COST_PER_PASSENGER_PER_LAP) + (num_cargo * COST_PER_CARGO_PER_LAP)
    cost = num_laps * cost_per_lap * efficiency_factor

    net_income = income - cost
    return net_income


In [ ]:
reference_passengers = 50
reference_cargo = np.floor(50/3)
reference_laps = 5
reference_battery = 100

# Calculate payload weight and volume
payload = payload_properties(reference_passengers, reference_cargo)
print(f"--- Payload Properties for {reference_passengers} Passengers ---")
print(f"Number of Cargo Pieces: {payload['num_cargo']}")
print(f"Total Weight: {payload['weight_oz']:.2f} oz")
print(f"Estimated Volume: {payload['estimated_volume_in3']:.2f} in³\n")

score = net_income(reference_passengers, reference_cargo, reference_laps)
print("--- Score Calculation Example ---")
print(f"Net Income for {reference_passengers} passengers and {reference_laps} laps: {score:.2f}\n")

In [ ]:
def mission2_score(num_passengers, num_cargo, num_laps, battery_capacity_wh):
    return 1 + (
        net_income(num_passengers, num_cargo, num_laps, battery_capacity_wh)
        /
        net_income(
            reference_passengers, reference_cargo,
            reference_laps, reference_battery,
        )
    )
    
def mission2_deviation(num_passengers, num_cargo, num_laps, battery_capacity_wh):
    return (
        mission2_score(num_passengers, num_cargo, num_laps, battery_capacity_wh) -
        mission2_score(
            reference_passengers, reference_cargo,
            reference_laps, reference_battery
        )
    )

In [ ]:
sensitivity_parameter_adj = np.linspace(0.7, 1.3)

plt.plot(
    sensitivity_parameter_adj,
    mission2_deviation(
        reference_passengers * sensitivity_parameter_adj,
        reference_cargo,
        reference_laps,
        reference_battery,
    ),
    color='b',
    label='Passenger Count',
)


plt.plot(
    sensitivity_parameter_adj,
    mission2_deviation(
        reference_passengers,
        reference_cargo * sensitivity_parameter_adj,
        reference_laps,
        reference_battery,
    ),    
    color='r',
    label='Cargo Amount',
)

plt.plot(
    sensitivity_parameter_adj,
    mission2_deviation(
        reference_passengers,
        reference_cargo,
        reference_laps * sensitivity_parameter_adj,
        reference_battery * sensitivity_parameter_adj,
    ),    
    color='g',
    label='Battery Capacity',
)

plt.plot(
    sensitivity_parameter_adj,
    mission2_deviation(
        reference_passengers,
        reference_cargo,
        reference_laps * sensitivity_parameter_adj**2,
        # Chatgpt said that range is proportional to the wingspan squared
        # and I don't feel like thinking freely right now
        reference_battery,
    ),    
    color='y',
    label='Wingspan',
)

plt.rcParams['text.usetex'] = True
plt.xlabel(r'\begin{center}Relative Parameter Change\\*\textbf{Figure 3: Mission 2 Scoring Sensitivty}\end{center}')
plt.ylabel("Relative Score Change")
plt.legend()

plt.show()

# Mission 3